# Stage 14: Deployment & Monitoring

Monitoring plan for the Stage 13 value model. Project-side deliverables: `project/docs/monitoring_plan.md`, `project/docs/handoff_plan.md`.

## 1) Reflection

**Model in scope.** The Stage 10a tier-value regression, packaged in Stage 13 as
`src/model.py` and served by `app.py` `POST /predict`. It predicts a card tier's
`log10(est_value_usd)` from `log10(odds_pack)` plus type flags.

**Risks if deployed.** (1) The label is a `rough_estimate_v0` placeholder, not real market
comps, so predictions can be confidently wrong. (2) Topps re-issues checklists often — a
renamed column or new tier type breaks the feature build silently. (3) Card values move
with player performance, so a once-fit model goes stale within weeks. (4) Stage 11 already
found a +0.23 residual bias on autographs, which could widen unnoticed.

**Monitoring across layers.**
- *Data:* SHA-256 of `card_tiers.csv` columns (alert on any change); `odds_pack` null rate
  (baseline ~7%, alert > 15%); feed freshness (alert if no checklist refresh in 14 days).
- *Model:* 4-week rolling MAE of `/predict` vs. incoming eBay sold comps (baseline 0.32
  log10-USD, alert > 0.45 for two weeks); auto-tier mean residual (alert > +0.35); rarity
  slope (retrain if outside [0.55, 0.80]).
- *System:* `/predict` p95 latency (alert > 250 ms); API 5xx rate (alert > 1% over 5 min);
  weekly `ev_report.csv` job success.
- *Business:* box configs flagged positive-EV, now 3 of 29 (alert if it swings by >= 2
  week-over-week with no matching price update).

**Ownership & handoffs.** The project owner (analyst) reviews the Model and Business metrics
weekly and approves every retrain and `model/model.pkl` rollback. Platform on-call owns the
System metrics and may restart `app.py` without approval, escalating Model/Data alerts to
the owner. Alerts go to the owner by email and `#aaa-cardshop-alerts`; issues are logged as
GitHub Issues labelled `monitoring`. Retrain triggers: >= 20 new real comps, rolling MAE
> 0.45 for two weeks, or a new product checklist.

## 2) Dashboard Sketch

One page, four rows (one per layer), refreshed hourly:

1. **Data** — line: `odds_pack` null rate over 30 days with the 15% alert line; tile:
   current schema hash vs expected (green/red); tile: days since last checklist refresh.
2. **Model** — line: 4-week rolling MAE vs the 0.45 threshold; bar: mean residual by
   `tier_group` (auto highlighted, +0.35 marker); gauge: rarity slope inside/outside
   [0.55, 0.80].
3. **System** — line: `/predict` p95 latency (250 ms line); line: 5xx rate; tile: last
   `ev_report.csv` refresh time + pass/fail.
4. **Business** — bar: positive-EV box count per week (3/29 baseline line); table: current
   buy/pass list from `ev_report.csv`.

Header strip: model version (`model.pkl` git sha), last retrain date, on-call name.

In [1]:
# Metric structure the plan is built from (layer -> [metric, ...]).
monitoring = {
    "data":     ["schema_hash", "odds_pack_null_rate", "feed_freshness_days"],
    "model":    ["rolling_mae_4wk", "auto_tier_residual", "rarity_slope"],
    "system":   ["predict_p95_latency_ms", "api_5xx_rate", "ev_report_job_ok"],
    "business": ["positive_ev_box_count"],
}
thresholds = {
    "odds_pack_null_rate": "> 15%",
    "rolling_mae_4wk": "> 0.45 for 2 wks",
    "auto_tier_residual": "> +0.35",
    "rarity_slope": "outside [0.55, 0.80]",
    "predict_p95_latency_ms": "> 250",
    "api_5xx_rate": "> 1% / 5 min",
    "positive_ev_box_count": "+/- 2 wk-over-wk",
}
{layer: [(m, thresholds.get(m, "any change / see plan")) for m in ms]
 for layer, ms in monitoring.items()}

{'data': [('schema_hash', 'any change / see plan'),
  ('odds_pack_null_rate', '> 15%'),
  ('feed_freshness_days', 'any change / see plan')],
 'model': [('rolling_mae_4wk', '> 0.45 for 2 wks'),
  ('auto_tier_residual', '> +0.35'),
  ('rarity_slope', 'outside [0.55, 0.80]')],
 'system': [('predict_p95_latency_ms', '> 250'),
  ('api_5xx_rate', '> 1% / 5 min'),
  ('ev_report_job_ok', 'any change / see plan')],
 'business': [('positive_ev_box_count', '+/- 2 wk-over-wk')]}